In [ ]:
import os
import pandas as pd


# 1. Mendapatkan path ke folder Desktop secara otomatis
folder_path = r'C:\protofolio\ecommerce_analytic\data bersih'

# 3. Membaca file menggunakan pandas
df_customers = pd.read_csv(os.path.join(folder_path, 'customers_clean.csv')).copy()

df_events = pd.read_csv(os.path.join(folder_path, 'events_clean.csv')).copy()

df_campaigns = pd.read_csv(os.path.join(folder_path, 'campaigns_clean.csv')).copy()

#transaction_col_to_use=['event_id','quantity','discount_applied','gross_revenue','net_revenue','refund_flag']
df_transactions = pd.read_csv(os.path.join(folder_path, 'transactions_clean.csv')).copy()

df_products = pd.read_csv(os.path.join(folder_path, 'products_clean.csv')).copy()

#d#f_customers.head()

left join table event+product

In [2]:
#left join table event+product
df_master=pd.merge(df_events,df_products,on='product_id',how='left')
df_master.head()

,event_id,timestamp,customer_id,session_id,event_type,product_id,device_type,traffic_source,campaign_id,page_category,session_duration_sec,experiment_group,created_at,category,brand,base_price,launch_date,is_premium
0,1,2021-01-14 13:35:43,43812,535101,view,1004.0,desktop,Email,43,PLP,115.1,Control,2021-01-14,Home,Brand_10,49.93,2021-08-31,0.0
1,2,2021-12-03 21:36:50,71340,96426,add_to_cart,986.0,desktop,Email,10,PDP,32.4,Variant_A,2021-12-03,Grocery,Brand_58,46.89,2022-04-10,0.0
2,3,2021-12-27 08:25:15,59540,220126,purchase,1630.0,mobile,Organic,0,PDP,190.7,Variant_A,2021-12-27,Grocery,Brand_76,14.58,2022-11-02,0.0
3,4,2022-01-22 15:06:54,3601,484555,add_to_cart,1532.0,desktop,Paid Search,30,Checkout,134.8,Variant_B,2022-01-22,Beauty,Brand_16,15.81,2022-08-17,0.0
4,5,2021-05-10 12:03:09,92735,60646,bounce,-1.0,desktop,Email,26,PLP,53.1,Variant_A,2021-05-10,NaN,NaN,NaN,NaN,NaN


In [3]:
# Cek berapa banyak product_id di events yang ada di tabel products
print(f"Product di events yang cocok: {df_events['product_id'].isin(df_products['product_id']).mean() * 100:.2f}%")

Product di events yang cocok: 89.98%


In [4]:
# Mencari baris di events yang product_id-nya TIDAK ADA di tabel products
df_missing = df_events[~df_events['product_id'].isin(df_products['product_id'])]
# Tampilkan 5 contoh ID yang bermasalah --> id -1 karena memang sbg place holder karena user event
print(df_missing['product_id'].unique()[:5])

[-1.]


In [5]:
df_master.head()

,event_id,timestamp,customer_id,session_id,event_type,product_id,device_type,traffic_source,campaign_id,page_category,session_duration_sec,experiment_group,created_at,category,brand,base_price,launch_date,is_premium
0,1,2021-01-14 13:35:43,43812,535101,view,1004.0,desktop,Email,43,PLP,115.1,Control,2021-01-14,Home,Brand_10,49.93,2021-08-31,0.0
1,2,2021-12-03 21:36:50,71340,96426,add_to_cart,986.0,desktop,Email,10,PDP,32.4,Variant_A,2021-12-03,Grocery,Brand_58,46.89,2022-04-10,0.0
2,3,2021-12-27 08:25:15,59540,220126,purchase,1630.0,mobile,Organic,0,PDP,190.7,Variant_A,2021-12-27,Grocery,Brand_76,14.58,2022-11-02,0.0
3,4,2022-01-22 15:06:54,3601,484555,add_to_cart,1532.0,desktop,Paid Search,30,Checkout,134.8,Variant_B,2022-01-22,Beauty,Brand_16,15.81,2022-08-17,0.0
4,5,2021-05-10 12:03:09,92735,60646,bounce,-1.0,desktop,Email,26,PLP,53.1,Variant_A,2021-05-10,NaN,NaN,NaN,NaN,NaN


#join master with customers

In [6]:
#join master with customers
df_master=pd.merge(df_master,df_customers, on='customer_id', how="left")
df_master.head()

,event_id,timestamp,customer_id,session_id,event_type,product_id,device_type,traffic_source,campaign_id,page_category,...,brand,base_price,launch_date,is_premium,signup_date,country,age,gender,loyalty_tier,acquisition_channel
0,1,2021-01-14 13:35:43,43812,535101,view,1004.0,desktop,Email,43,PLP,...,Brand_10,49.93,2021-08-31,0.0,2022-05-20,DE,58,Male,Bronze,Email
1,2,2021-12-03 21:36:50,71340,96426,add_to_cart,986.0,desktop,Email,10,PDP,...,Brand_58,46.89,2022-04-10,0.0,2021-01-29,IN,21,Female,Bronze,Referral
2,3,2021-12-27 08:25:15,59540,220126,purchase,1630.0,mobile,Organic,0,PDP,...,Brand_76,14.58,2022-11-02,0.0,2021-11-26,IN,39,Female,Silver,Organic
3,4,2022-01-22 15:06:54,3601,484555,add_to_cart,1532.0,desktop,Paid Search,30,Checkout,...,Brand_16,15.81,2022-08-17,0.0,2022-05-03,UK,29,Female,Bronze,Email
4,5,2021-05-10 12:03:09,92735,60646,bounce,-1.0,desktop,Email,26,PLP,...,NaN,NaN,NaN,NaN,2022-09-17,UK,39,Female,Bronze,Email


In [7]:
print(f"Jumlah baris df_events awal: {len(df_events)}")
print(f"Jumlah baris df_master setelah merge: {len(df_master)}")

if len(df_master) > len(df_events):
    print("WARNING: Data meledak! Ada duplikat customer_id di tabel customers.")
else:
    print("OK: Jumlah baris konsisten.")

Jumlah baris df_events awal: 2000000
Jumlah baris df_master setelah merge: 2000000
OK: Jumlah baris konsisten.


In [8]:
# Cek berapa banyak customer_id di master
match_rate = df_master['customer_id'].isin(df_customers['customer_id']).mean() * 100
print(f"Customer di events yang punya profil: {match_rate:.2f}%")

Customer di events yang punya profil: 100.00%


In [9]:
df_master.head()

,event_id,timestamp,customer_id,session_id,event_type,product_id,device_type,traffic_source,campaign_id,page_category,...,brand,base_price,launch_date,is_premium,signup_date,country,age,gender,loyalty_tier,acquisition_channel
0,1,2021-01-14 13:35:43,43812,535101,view,1004.0,desktop,Email,43,PLP,...,Brand_10,49.93,2021-08-31,0.0,2022-05-20,DE,58,Male,Bronze,Email
1,2,2021-12-03 21:36:50,71340,96426,add_to_cart,986.0,desktop,Email,10,PDP,...,Brand_58,46.89,2022-04-10,0.0,2021-01-29,IN,21,Female,Bronze,Referral
2,3,2021-12-27 08:25:15,59540,220126,purchase,1630.0,mobile,Organic,0,PDP,...,Brand_76,14.58,2022-11-02,0.0,2021-11-26,IN,39,Female,Silver,Organic
3,4,2022-01-22 15:06:54,3601,484555,add_to_cart,1532.0,desktop,Paid Search,30,Checkout,...,Brand_16,15.81,2022-08-17,0.0,2022-05-03,UK,29,Female,Bronze,Email
4,5,2021-05-10 12:03:09,92735,60646,bounce,-1.0,desktop,Email,26,PLP,...,NaN,NaN,NaN,NaN,2022-09-17,UK,39,Female,Bronze,Email


#join campaign ke master

In [10]:
#join campaign ke master

df_master=pd.merge(df_master,df_campaigns,on='campaign_id',how='left')

df_master.head()

,event_id,timestamp,customer_id,session_id,event_type,product_id,device_type,traffic_source,campaign_id,page_category,...,loyalty_tier,acquisition_channel,channel,objective,start_date,end_date,target_segment,expected_uplift,duration_days,duration_days_int
0,1,2021-01-14 13:35:43,43812,535101,view,1004.0,desktop,Email,43,PLP,...,Bronze,Email,Paid Search,Cross-sell,2023-03-12,2023-04-18,Deal Seekers,0.078,37 days,37.0
1,2,2021-12-03 21:36:50,71340,96426,add_to_cart,986.0,desktop,Email,10,PDP,...,Bronze,Referral,Email,Cross-sell,2022-09-01,2022-10-22,Churn Risk,0.034,51 days,51.0
2,3,2021-12-27 08:25:15,59540,220126,purchase,1630.0,mobile,Organic,0,PDP,...,Silver,Organic,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,2022-01-22 15:06:54,3601,484555,add_to_cart,1532.0,desktop,Paid Search,30,Checkout,...,Bronze,Email,Social,Acquisition,2023-11-04,2024-01-06,Deal Seekers,0.052,63 days,63.0
4,5,2021-05-10 12:03:09,92735,60646,bounce,-1.0,desktop,Email,26,PLP,...,Bronze,Email,Email,Reactivation,2023-08-13,2023-11-01,Deal Seekers,0.037,80 days,80.0


In [11]:
# Cek berapa banyak campaign_id di master
print(f"campaign di events yang cocok: {df_master['campaign_id'].isin(df_campaigns['campaign_id']).mean() * 100:.2f}%")

campaign di events yang cocok: 49.99%


In [12]:
#df_master['channel'].head()
# Mengisi kolom channel  yang kosong dengan label 'Organik'
df_master['channel'] = df_master['channel'].fillna('Organic/Direct')
df_master['campaign_id'] = df_master['campaign_id'].fillna(0) # 0 berarti tidak ada ID

# Cek channel mana yang paling sering menghasilkan event
df_master['channel'].value_counts()

channel
Organic/Direct    1000251
Affiliate          220077
Email              219959
Paid Search        219726
Display            180397
Social             159590
Name: count, dtype: int64

In [13]:
df_master.head()

,event_id,timestamp,customer_id,session_id,event_type,product_id,device_type,traffic_source,campaign_id,page_category,...,loyalty_tier,acquisition_channel,channel,objective,start_date,end_date,target_segment,expected_uplift,duration_days,duration_days_int
0,1,2021-01-14 13:35:43,43812,535101,view,1004.0,desktop,Email,43,PLP,...,Bronze,Email,Paid Search,Cross-sell,2023-03-12,2023-04-18,Deal Seekers,0.078,37 days,37.0
1,2,2021-12-03 21:36:50,71340,96426,add_to_cart,986.0,desktop,Email,10,PDP,...,Bronze,Referral,Email,Cross-sell,2022-09-01,2022-10-22,Churn Risk,0.034,51 days,51.0
2,3,2021-12-27 08:25:15,59540,220126,purchase,1630.0,mobile,Organic,0,PDP,...,Silver,Organic,Organic/Direct,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,2022-01-22 15:06:54,3601,484555,add_to_cart,1532.0,desktop,Paid Search,30,Checkout,...,Bronze,Email,Social,Acquisition,2023-11-04,2024-01-06,Deal Seekers,0.052,63 days,63.0
4,5,2021-05-10 12:03:09,92735,60646,bounce,-1.0,desktop,Email,26,PLP,...,Bronze,Email,Email,Reactivation,2023-08-13,2023-11-01,Deal Seekers,0.037,80 days,80.0


In [14]:
# save master_cleand_data
df_master.to_csv(r'C:\protofolio\ecommerce_analytic\data bersih\master_clean.csv', index=False)
print("File sudah tersimpan!")

File sudah tersimpan!
